<a id="top"></a>
<div class="list-group" id="list-tab" role="tablist">
<h1 class="list-group-item list-group-item-action active" data-toggle="list" style='background:#005097; border:0' role="tab" aria-controls="home"><center>Hyperparameter Optimization</center></h1>

# Hyperparameter optimization

In machine learning, a **hyperparameter** is a value that controls the learning process of a model, but it is not directly learned from data. Hyperparameters define the inductive bias of the learning algorithm and strongly influence the complexity of the model that will be produced.

Some common examples are:

* the learning rate in gradient-based optimization,
* the regularization strength in linear or logistic regression,
* the value of $k$ in $k$-NN,
* the maximum depth of a decision tree,
* the number of trees in a random forest,
* the kernel and penalty parameter in an SVM.

The best hyperparameter setting usually depends on the dataset. Because of that, we need a **search procedure** that tries multiple configurations and an **evaluation procedure** that tells us which configuration generalizes better.

## What are we actually optimizing?

This is the key connection with the lecture notes. Hyperparameter search does **not** try to minimize the training error directly. Instead, it searches for the configuration that maximizes a validation metric estimated with cross-validation.

In practical terms, every tuning procedure in this notebook follows the same logic:

1. define a search space for the hyperparameters;
2. train several candidate models;
3. evaluate each candidate with cross-validation;
4. keep the configuration with the best validation score.

The main difference between the methods is how they explore the search space.

**Grid search** first defines a grid of hyperparameter combinations. The amount of hyperparameters defines the number of dimensions in the grid. Then, each combination is tested.

The image below ([source](https://www.researchgate.net/publication/271771573_TuPAQ_An_Efficient_Planner_for_Large-scale_Predictive_Analytic_Queries/figures?lo=1)) illustrates grid search for a model with two hyperparameters.

<p align="center">
<img src="https://www.researchgate.net/profile/Michael_Jordan13/publication/271771573/figure/fig4/AS:668513593217027@1536397469229/Illustration-of-naive-grid-search-for-a-single-model-family-with-two-hyperparameters_W640.jpg">
</p>

**Random search** tests only a subset of the possible combinations. Instead of enumerating the whole grid, it samples candidate configurations from preconfigured ranges.

This matters because, as discussed in the lecture notes, not all hyperparameters have the same influence on performance. When only a few dimensions matter a lot, random search may cover promising regions faster than grid search under the same computational budget.

The image below ([source](https://community.alteryx.com/t5/Data-Science/Hyperparameter-Tuning-Black-Magic/ba-p/449289)) illustrates that idea.

<p align="center">
<img src="https://pvsmt99345.i.lithium.com/t5/image/serverpage/image-id/74545i97245FDAA10376E9/image-size/large?v=1.0&px=999">
</p>

In Scikit-Learn, the classes [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) and [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) implement these two strategies.

### Class GridSearchCV

The first example uses `GridSearchCV` with an SVM classifier on the Iris dataset. The goal is to search over three hyperparameters:

* `C`, which controls regularization strength;
* `kernel`, which changes the type of decision boundary;
* `gamma`, which matters for non-linear kernels such as `rbf`.

When reading the code below, focus on four objects:

* `svc`: the learning algorithm being tuned;
* `param_grid`: the search space;
* `cv=5`: the inner validation procedure used to compare candidates;
* `scoring='accuracy'`: the metric that defines what "best" means.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

# Keep a test split outside the search procedure
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

svc = SVC()

param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid_search = GridSearchCV(svc, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print("Best parameters found:", grid_search.best_params_)
print("Best cross-validation score: {:.2f}".format(grid_search.best_score_))

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\nClassification report on the test set:\n")
print(classification_report(y_test, y_pred))


A few comments about the result:

* `best_score_` is the average score obtained in the internal cross-validation of `GridSearchCV`.
* The test set is used only after the search is over. This separation is important to avoid optimistic estimates.
* `best_estimator_` is a model refit on the full training split using the best hyperparameters found during the search.

This is exactly the workflow emphasized in the lecture notes: model selection is driven by validation performance, and the test split is kept for a final independent check.

### Class RandomizedSearchCV

`RandomizedSearchCV` exposes an explicit budget through `n_iter`: instead of evaluating all combinations, it evaluates only a fixed number of sampled configurations.

The code below uses a large list of possible values for `C`. This is intentional: it makes the contrast with grid search easier to see. Grid search tests all values in the list, while random search tests only `n_iter` sampled values.

Pedagogically, the point is not that one timing number will always be smaller than the other. The important point is that random search gives us direct control over the search budget.

In [ ]:
from timeit import default_timer as timer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.svm import LinearSVC


def run_grid_search(x, y, candidate_values, kfold):
    param_grid = {'C': candidate_values}
    cv = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=42)
    search = GridSearchCV(LinearSVC(dual=False, random_state=42), param_grid=param_grid, cv=cv, n_jobs=4, verbose=1)
    return search.fit(x, y)


def run_random_search(x, y, candidate_values, kfold, n_iter):
    param_grid = {'C': candidate_values}
    cv = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=42)
    search = RandomizedSearchCV(
        LinearSVC(dual=False, random_state=42),
        param_distributions=param_grid,
        cv=cv,
        n_jobs=4,
        verbose=1,
        n_iter=n_iter,
        random_state=42
    )
    return search.fit(x, y)


iris = load_iris()
X = iris.data
y = iris.target

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

candidate_values = [i / 1000 for i in range(1, 1000)]
candidate_values.extend([i for i in range(1, 101)])

start = timer()
random_search = run_random_search(x_train, y_train, candidate_values, kfold=3, n_iter=100)
random_duration = timer() - start

print('RandomizedSearchCV')
print('Best CV accuracy:', random_search.best_score_)
print('Test set score:  ', random_search.score(x_test, y_test))
print('Best parameters: ', random_search.best_params_)
print('Elapsed time:    ', random_duration)
print()

start = timer()
grid_search = run_grid_search(x_train, y_train, candidate_values, kfold=3)
grid_duration = timer() - start

print('GridSearchCV')
print('Best CV accuracy:', grid_search.best_score_)
print('Test set score:  ', grid_search.score(x_test, y_test))
print('Best parameters: ', grid_search.best_params_)
print('Elapsed time:    ', grid_duration)


This example should be read with some care:

* the dataset is small, so absolute timings are not the main lesson;
* the key difference is that grid search is exhaustive on the discrete grid, while random search spends only a fixed budget of trials;
* in high-dimensional spaces, that budget control can be much more important than exhaustive coverage.

This is the natural transition to the next idea from the lecture notes: if exhaustive search is expensive and naive random search wastes trials, can we choose new trials more intelligently?

# Optuna

Optuna is a hyperparameter optimization framework built around the idea of an **objective function**. Instead of enumerating a predefined grid, we write a function that:

* receives a `trial` object,
* asks the trial for candidate hyperparameter values,
* trains and evaluates the corresponding model,
* returns a score to maximize or minimize.

This makes the code more flexible and, more importantly, allows the search strategy to adapt based on previous trials.

## Why Optuna is different from Grid Search and Random Search

If no sampler is specified, Optuna uses **TPE (Tree-structured Parzen Estimator)** by default. The practical intuition is simple: after observing previous trials, it tries to sample new configurations in regions that look more promising.

That is the conceptual bridge with the lecture notes:

* grid search is exhaustive on a predefined grid;
* random search is budgeted but memoryless;
* Optuna can use the history of the search to guide future trials.

The notebook below keeps the default sampler to emphasize the basic workflow before discussing more advanced sampler customization.

## Demo: Optuna with nested cross-validation

This example is more realistic than the Iris examples above. We have a tabular dataset with numerical and categorical variables, so the model is built as a pipeline with preprocessing plus a classifier.

The structure of the code is important:

* the **inner loop** uses Optuna to choose hyperparameters;
* the **outer loop** estimates the generalization performance;
* the score optimized by Optuna is the mean F1 obtained inside the inner loop.

This is a nested cross-validation setting. It is more computationally expensive, but it gives a much cleaner estimate of model performance.

In [ ]:
# pip install optuna

In [ ]:
import pandas as pd

data = pd.read_csv('../data/aug_train.csv')

# Split features and target
X = data.drop(columns=['id', 'Response'])
y = data['Response']

# Define categorical and numerical features
categorical_features = ['Gender', 'Vehicle_Age', 'Vehicle_Damage']
numerical_features = X.columns.difference(categorical_features)

print('Dataset shape:', X.shape)
print('Positive class rate:', y.mean())


Before reading the next code cell, notice the role of each object:

* `preprocessor` prevents leakage by placing preprocessing inside the pipeline;
* `objective(trial)` defines the search space and returns the mean inner-CV F1;
* `study.optimize(...)` performs the search for one outer fold;
* `outer_scores` stores the performance on the held-out outer fold.

This means that the score printed at the end is **not** the score of a single final model. It is an estimate of expected generalization performance produced by nested CV.

In [ ]:
import numpy as np
import optuna
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(), categorical_features)
])

N_TRIALS = 10
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_scores = []
best_params_per_fold = []

for fold_id, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    def objective(trial):
        model = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', RandomForestClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 300),
                max_depth=trial.suggest_int('max_depth', 3, 20),
                min_samples_split=trial.suggest_int('min_samples_split', 2, 10),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
                random_state=42
            ))
        ])

        inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        scores = cross_val_score(model, X_train, y_train, cv=inner_cv, scoring='f1', n_jobs=-1)
        return np.mean(scores)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS)

    best_params = study.best_params
    best_params_per_fold.append(best_params)

    final_model_for_this_fold = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(**best_params, random_state=42))
    ])
    final_model_for_this_fold.fit(X_train, y_train)
    y_pred = final_model_for_this_fold.predict(X_test)
    score = f1_score(y_test, y_pred)
    outer_scores.append(score)

    print(f'Fold {fold_id}: outer F1 = {score:.4f}, best params = {best_params}')

print('\nNested CV F1 scores:', outer_scores)
print('Mean F1 score:', np.mean(outer_scores))


## What nested CV gives us, and what it does not give us

At this point, the notebook has produced an estimate of generalization performance. That is the main purpose of nested CV.

However, nested CV does **not** automatically produce a single final model ready for deployment. After evaluation, a common next step is to run a new hyperparameter search on the full dataset and then refit one final model using the best configuration found there.

This is the same distinction discussed in the lecture notes and in the more complete `model_selection.ipynb`: one procedure estimates performance, the other produces the final model.

In [ ]:
%%time

def objective_full_data(trial):
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=trial.suggest_int('n_estimators', 50, 300),
            max_depth=trial.suggest_int('max_depth', 3, 20),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 10),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 10),
            random_state=42
        ))
    ])

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=cv, scoring='f1', n_jobs=-1)
    return np.mean(scores)


study_full_data = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_full_data.optimize(objective_full_data, n_trials=N_TRIALS)

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(**study_full_data.best_params, random_state=42))
])
final_model.fit(X, y)

print('Best hyperparameters on the full dataset:', study_full_data.best_params)
print('The final model has now been refit on all available data.')


## Suggested exercises

To connect this notebook even more strongly with the lecture notes, good follow-up exercises are:

1. replace the default TPE sampler with `RandomSampler` and compare the best score reached after the same number of trials;
2. increase `N_TRIALS` and observe whether the outer-fold scores become more stable;
3. inspect `best_params_per_fold` and discuss why the "best" hyperparameters may vary across folds;
4. change the optimization metric from F1 to accuracy and analyze how the selected configurations change.

These variations make the search procedure itself part of the learning objective, instead of treating tuning as a black box.